In [3]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


PROJECT_ROOT = Path.cwd()

# If the notebook is running from notebooks/, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
TEST_DIR = PROJECT_ROOT / "data" / "test"

print("Project root:", PROJECT_ROOT)
print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

Project root: /Users/ankitraj/Desktop/Amazon_Ml_challenge26
Train directory: /Users/ankitraj/Desktop/Amazon_Ml_challenge26/data/train
Test directory: /Users/ankitraj/Desktop/Amazon_Ml_challenge26/data/test


In [5]:
print("Train files:")
for path in sorted(TRAIN_DIR.iterdir()):
    print(" ", path.name)

print("\nTest files:")
for path in sorted(TEST_DIR.iterdir()):
    print(" ", path.name)

Train files:
  .DS_Store
  train_ground_truth.tsv
  train_source1.tsv
  train_source2.tsv
  train_source3.tsv

Test files:
  .DS_Store
  test_source1.tsv
  test_source2.tsv
  test_source3.tsv


In [7]:
import sys

# Add the project's src/ directory to Python's import path
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Using source directory:", SRC_DIR)

Using source directory: /Users/ankitraj/Desktop/Amazon_Ml_challenge26/src


In [8]:
from business_entity_resol.io.reader import read_source_file

print("business_entity_resol imported successfully.")

business_entity_resol imported successfully.


In [9]:
train_source1 = read_source_file(
    TRAIN_DIR / "train_source1.tsv"
)

train_source2 = read_source_file(
    TRAIN_DIR / "train_source2.tsv"
)

train_source3 = read_source_file(
    TRAIN_DIR / "train_source3.tsv"
)

test_source1 = read_source_file(
    TEST_DIR / "test_source1.tsv"
)

test_source2 = read_source_file(
    TEST_DIR / "test_source2.tsv"
)

test_source3 = read_source_file(
    TEST_DIR / "test_source3.tsv"
)

print("All six source files loaded successfully.")

All six source files loaded successfully.


In [10]:
train_sources = {
    "train_source1": train_source1,
    "train_source2": train_source2,
    "train_source3": train_source3,
}

test_sources = {
    "test_source1": test_source1,
    "test_source2": test_source2,
    "test_source3": test_source3,
}

for name, df in {**train_sources, **test_sources}.items():
    print(f"{name}: {df.shape}")

train_source1: (2206821, 4)
train_source2: (5034616, 4)
train_source3: (5285603, 4)
test_source1: (1732544, 4)
test_source2: (4887273, 4)
test_source3: (5082316, 4)


In [11]:
def basic_source_summary(sources):
    rows = []

    for source_name, df in sources.items():
        rows.append(
            {
                "source": source_name,
                "rows": len(df),
                "columns": len(df.columns),
                "unique_entity_ids": df["entity_id"].nunique(),
            }
        )

    return pd.DataFrame(rows)


train_basic = basic_source_summary(train_sources)
test_basic = basic_source_summary(test_sources)

print("TRAIN")
display(train_basic)

print("\nTEST")
display(test_basic)

TRAIN


,source,rows,columns,unique_entity_ids
0,train_source1,2206821,4,2206821
1,train_source2,5034616,4,5034616
2,train_source3,5285603,4,5285603



TEST


,source,rows,columns,unique_entity_ids
0,test_source1,1732544,4,1732544
1,test_source2,4887273,4,4887273
2,test_source3,5082316,4,5082316


In [12]:
def missing_value_summary(sources):
    rows = []

    for source_name, df in sources.items():
        total = len(df)

        for column in [
            "business_name",
            "business_address",
            "country",
        ]:
            missing_count = (
                df[column].isna()
                | df[column].astype("string").str.strip().eq("")
            ).sum()

            rows.append(
                {
                    "source": source_name,
                    "field": column,
                    "missing_count": int(missing_count),
                    "missing_percentage": round(
                        missing_count / total * 100,
                        4,
                    ),
                }
            )

    return pd.DataFrame(rows)


train_missing = missing_value_summary(train_sources)
test_missing = missing_value_summary(test_sources)

print("TRAIN MISSING VALUES")
display(train_missing)

print("TEST MISSING VALUES")
display(test_missing)

TRAIN MISSING VALUES


,source,field,missing_count,missing_percentage
0,train_source1,business_name,0,0.0000
1,train_source1,business_address,0,0.0000
2,train_source1,country,0,0.0000
3,train_source2,business_name,0,0.0000
4,train_source2,business_address,168967,3.3561
5,train_source2,country,0,0.0000
6,train_source3,business_name,0,0.0000
7,train_source3,business_address,175916,3.3282
8,train_source3,country,0,0.0000


TEST MISSING VALUES


,source,field,missing_count,missing_percentage
0,test_source1,business_name,0,0.0000
1,test_source1,business_address,0,0.0000
2,test_source1,country,0,0.0000
3,test_source2,business_name,0,0.0000
4,test_source2,business_address,129408,2.6479
5,test_source2,country,0,0.0000
6,test_source3,business_name,0,0.0000
7,test_source3,business_address,136098,2.6779
8,test_source3,country,0,0.0000


In [13]:
def duplicate_summary(sources):
    rows = []

    for source_name, df in sources.items():
        rows.append(
            {
                "source": source_name,
                "duplicate_entity_id": int(
                    df["entity_id"].duplicated().sum()
                ),
                "duplicate_business_name": int(
                    df["business_name"].duplicated().sum()
                ),
                "duplicate_business_address": int(
                    df["business_address"].duplicated().sum()
                ),
            }
        )

    return pd.DataFrame(rows)


train_duplicates = duplicate_summary(train_sources)
test_duplicates = duplicate_summary(test_sources)

print("TRAIN DUPLICATES")
display(train_duplicates)

print("TEST DUPLICATES")
display(test_duplicates)

TRAIN DUPLICATES


,source,duplicate_entity_id,duplicate_business_name,duplicate_business_address
0,train_source1,0,667592,76215
1,train_source2,0,632607,697354
2,train_source3,0,633994,652838


TEST DUPLICATES


,source,duplicate_entity_id,duplicate_business_name,duplicate_business_address
0,test_source1,0,493677,55061
1,test_source2,0,576232,662489
2,test_source3,0,560387,625880


In [14]:
def country_summary(sources):
    rows = []

    for source_name, df in sources.items():
        counts = (
            df["country"]
            .fillna("")
            .astype(str)
            .str.strip()
            .replace("", "<MISSING>")
            .value_counts()
        )

        for country, count in counts.items():
            rows.append(
                {
                    "source": source_name,
                    "country": country,
                    "count": int(count),
                }
            )

    return pd.DataFrame(rows)


train_country_summary = country_summary(train_sources)
test_country_summary = country_summary(test_sources)

print("TRAIN COUNTRIES")
display(train_country_summary)

print("TEST COUNTRIES")
display(test_country_summary)

TRAIN COUNTRIES


,source,country,count
0,train_source1,US,1323633
1,train_source1,India,883188
2,train_source2,US,3016817
3,train_source2,India,2017799
4,train_source3,US,3170056
5,train_source3,India,2115547


TEST COUNTRIES


,source,country,count
0,test_source1,India,809986
1,test_source1,US,663106
2,test_source1,France,259452
3,test_source2,India,2312565
4,test_source2,US,1871330
5,test_source2,France,703378
6,test_source3,India,2405000
7,test_source3,US,1945701
8,test_source3,France,731615


In [15]:
from business_entity_resol.io.ground_truth import read_ground_truth

ground_truth_path = TRAIN_DIR / "train_ground_truth.tsv"

ground_truth = read_ground_truth(ground_truth_path)

print("Source-1 entities in ground truth:", len(ground_truth))

Source-1 entities in ground truth: 2206821


In [16]:
match_counts = pd.Series(
    {
        source1_id: len(matches)
        for source1_id, matches in ground_truth.items()
    },
    name="match_count",
)

match_distribution = pd.DataFrame(
    {
        "category": [
            "zero_matches",
            "one_match",
            "two_matches",
            "three_or_more_matches",
        ],
        "count": [
            int((match_counts == 0).sum()),
            int((match_counts == 1).sum()),
            int((match_counts == 2).sum()),
            int((match_counts >= 3).sum()),
        ],
    }
)

display(match_distribution)

,category,count
0,zero_matches,123247
1,one_match,119157
2,two_matches,375212
3,three_or_more_matches,1589205


In [17]:
s2_entity_count = 0
s3_entity_count = 0
both_entity_count = 0

for matches in ground_truth.values():
    has_s2 = any(x.startswith("S2-") for x in matches)
    has_s3 = any(x.startswith("S3-") for x in matches)

    s2_entity_count += has_s2
    s3_entity_count += has_s3
    both_entity_count += has_s2 and has_s3

ground_truth_summary = pd.DataFrame(
    {
        "metric": [
            "Source-1 entities",
            "Entities matched to S2",
            "Entities matched to S3",
            "Entities matched to both S2 and S3",
        ],
        "count": [
            len(ground_truth),
            s2_entity_count,
            s3_entity_count,
            both_entity_count,
        ],
    }
)

display(ground_truth_summary)

,metric,count
0,Source-1 entities,2206821
1,Entities matched to S2,1919076
2,Entities matched to S3,1940545
3,Entities matched to both S2 and S3,1776047


# Findings

## Dataset size

The dataset is large, with more records in Sources 2 and 3 than in the
Source-1 reference data.

### Training data

- Source 1: 2,206,821 records
- Source 2: 5,034,616 records
- Source 3: 5,285,603 records

### Test data

- Source 1: 1,732,544 records
- Source 2: 4,887,273 records
- Source 3: 5,082,316 records

Every source contains the four expected columns:
`entity_id`, `business_name`, `business_address`, and `country`.

The number of unique entity IDs equals the number of rows in every
source. No duplicate entity IDs were observed.

## Missing values

Business names and countries are complete across all train and test
sources.

Missing addresses occur only in Sources 2 and 3.

### Training

- Source 1: 0 missing addresses
- Source 2: 168,967 missing addresses (3.3561%)
- Source 3: 175,916 missing addresses (3.3282%)

### Test

- Source 1: 0 missing addresses
- Source 2: 129,408 missing addresses (2.6479%)
- Source 3: 136,098 missing addresses (2.6779%)

Therefore, downstream matching cannot assume that an address is always
available. Name-based and other representations remain important for
records with missing addresses.

## Duplicate analysis

No duplicate entity IDs occur in any source.

Business names and addresses, however, are frequently duplicated.

### Training duplicate values

- Source 1:
  - business names: 667,592
  - business addresses: 76,215
- Source 2:
  - business names: 632,607
  - business addresses: 697,354
- Source 3:
  - business names: 633,994
  - business addresses: 652,838

### Test duplicate values

- Source 1:
  - business names: 493,677
  - business addresses: 55,061
- Source 2:
  - business names: 576,232
  - business addresses: 662,489
- Source 3:
  - business names: 560,387
  - business addresses: 625,880

Duplicate business names or addresses therefore cannot be treated as
duplicate entities. Entity IDs remain the record identifiers.

## Ground truth

The training ground truth contains all 2,206,821 Source-1 entities.

Match-count distribution:

- Zero matches: 123,247
- Exactly one match: 119,157
- Exactly two matches: 375,212
- Three or more matches: 1,589,205

Source coverage:

- Source-1 entities matched to Source 2: 1,919,076
- Source-1 entities matched to Source 3: 1,940,545
- Source-1 entities matched to both Source 2 and Source 3: 1,776,047

This confirms that entity resolution is not a simple one-to-one mapping.
A Source-1 entity may have zero, one, or multiple corresponding records.

## Country analysis

Training contains:

- US
- India

Test contains:

- India
- US
- France

### Training

Source 1:
- US: 1,323,633
- India: 883,188

Source 2:
- US: 3,016,817
- India: 2,017,799

Source 3:
- US: 3,170,056
- India: 2,115,547

### Test

Source 1:
- India: 809,986
- US: 663,106
- France: 259,452

Source 2:
- India: 2,312,565
- US: 1,871,330
- France: 703,378

Source 3:
- India: 2,405,000
- US: 1,945,701
- France: 731,615

France occurs in test but not training. Preprocessing must therefore
remain country-agnostic and must not assume that the set of countries
seen during training is exhaustive.

## Preprocessing implications

1. Raw business names and addresses should always be preserved alongside
   normalized representations.

2. Missing addresses must be handled explicitly because approximately
   2.6-3.4% of Source-2/Source-3 records have no address.

3. Duplicate names and addresses are common and must not be interpreted
   as duplicate entities.

4. Multiple representations such as normalized text, compact text,
   tokens, and numeric address components are useful because no single
   representation is sufficient for all records.

5. Country handling must support previously unseen values because France
   occurs in test despite being absent from training.

6. Ground truth is inherently one-to-many: most Source-1 entities have
   multiple matching records, while some have no match.

In [18]:
source1_ids = set(train_source1["entity_id"])
ground_truth_ids = set(ground_truth)

missing_from_ground_truth = source1_ids - ground_truth_ids
unknown_ground_truth_ids = ground_truth_ids - source1_ids

print("Source-1 IDs missing from ground truth:",
      len(missing_from_ground_truth))

print("Ground-truth IDs absent from Source 1:",
      len(unknown_ground_truth_ids))

Source-1 IDs missing from ground truth: 0
Ground-truth IDs absent from Source 1: 0
